# I05 - Transfer Learning and Fine-tuning

**MAI Alignment:** COMPSCI 714 (Lectures 7-1, 7-2), COMPSCI 703, COMPSYS 721 | **Prerequisite:** [B09 - CNNs](../Basic/B09%20-%20Convolutional%20Neural%20Networks.ipynb) | **Next Level:** [A01 - Fine-tuning LLMs](../Advanced/A01%20-%20Fine-tuning%20Large%20Language%20Models.ipynb)

**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand the formal definition of transfer learning (source/target domains and tasks)
2. Navigate the full transfer learning taxonomy: Model Fine-tuning, Conservation Learning, Layer Transfer, Multitask Learning, Domain-Adversarial Training, Zero/Few-shot Learning, and Self-supervised Learning
3. Use pre-trained models from ImageNet for feature extraction
4. Master fine-tuning strategies including Conservation Learning constraints
5. Implement Domain-Adversarial Training concepts
6. Apply Zero-shot and Few-shot learning techniques

---

## Prerequisites

- Completed I04 (Advanced CNN Architectures)
- Understanding of CNNs and training
- Familiarity with different architectures

---

> 📚 **COMPSCI 714 Alignment:** This notebook directly covers the content from Lectures 7-1 and 7-2 (Transfer Learning Methods). The formal taxonomy and definitions match the course slides exactly.

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    ResNet50, VGG16, MobileNetV2, EfficientNetB0
)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. What is Transfer Learning? (COMPSCI 714 Formal Definition)

### Formal Definition

From COMPSCI 714 Lecture 7-2:

> Given a **source domain** 𝒟ₛ with corresponding learning task 𝒯ₛ, and a **target domain** 𝒟ₜ with corresponding learning task 𝒯ₜ, **transfer learning** is the process of improving the target predictive function fₜ(·) in 𝒟ₜ by using the knowledge in 𝒟ₛ and 𝒯ₛ, **where 𝒟ₛ ≠ 𝒟ₜ or 𝒯ₛ ≠ 𝒯ₜ**.

**Example:**
- Source domain: 10 million movie reviews in English → sentiment analysis (positive/negative)
- Target domain: 1 million clothes reviews in Chinese → sentiment analysis (rating 1–10)

### The Full Transfer Learning Taxonomy

The choice of method depends on whether source/target data is labelled or unlabelled:

| | **Source: Labelled** | **Source: Unlabelled** |
|---|---|---|
| **Target: Labelled** | Model Fine-tuning, Multitask Learning | Self-taught Learning |
| **Target: Unlabelled** | Domain-Adversarial Training, Zero/Few-shot Learning | Self-taught Clustering |

### Why Transfer Learning?
- Limited training data in the target domain
- Expensive to train from scratch
- Pre-trained models learn general features
- Faster convergence and better performance

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Resize images to 96x96 (closer to ImageNet size)
x_train_resized = tf.image.resize(x_train, [96, 96]).numpy()
x_test_resized = tf.image.resize(x_test, [96, 96]).numpy()

# Use subset for faster training
x_train_small = x_train_resized[:5000]
y_train_small = y_train[:5000]

print(f"Training samples: {x_train_small.shape}")
print(f"Test samples: {x_test_resized.shape}")

# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

## 2. Model Fine-tuning and Conservation Learning (COMPSCI 714 Lecture 7-2)

### Model Fine-tuning Task Setup

- **Source data** (xˢ, yˢ): A **large** amount — e.g., audio data from many speakers (10K+ hours)
- **Target data** (xᵗ, yᵗ): **Very little** — e.g., only 3 examples from a specific user
- **Solution:** Train a model on source data, then **fine-tune** on target data
- **Challenge:** Only limited target data → be careful about **overfitting**

### Conservation Learning

To prevent overfitting when fine-tuning with limited target data, we apply **Conservation Learning** constraints:

**Goal:** Ensure the difference between the new and old models is not too large.

**Two constraints:**
1. **Output constraint:** With the same input, the new model output and old model output should be close
2. **Parameter constraint:** The parameters of new and old model should be close (e.g., measured by L2 norm)

This is equivalent to adding a regularization term to the fine-tuning loss:

```
L_total = L_task(target data) + λ · L_conservation(new model vs old model)
```

### Layer Transfer

Instead of fine-tuning all layers, we can **copy (transfer) some layers** from the source model:

- **Speech:** Usually copy the **last** few layers (task-specific representations)
- **Images:** Usually copy the **first** few layers (generic low-level features like edges, textures)

**Two strategies after layer transfer:**
1. **Only train the rest of the layers** using target data (prevents overfitting)
2. **Fine-tune the whole network** (only if there is sufficient target data)

In [ ]:
# Load pre-trained ResNet50 (without top classification layer)
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)

# Freeze all layers in base model
base_model.trainable = False

print(f"Base model has {len(base_model.layers)} layers")
print(f"Trainable: {base_model.trainable}")
print(f"\nBase model output shape: {base_model.output_shape}")

In [ ]:
# Build feature extraction model
def create_feature_extraction_model(base_model, num_classes=10):
    """Create model using pre-trained base as feature extractor"""
    
    inputs = keras.Input(shape=(96, 96, 3))
    
    # Pre-trained base (frozen)
    x = base_model(inputs, training=False)
    
    # Global pooling
    x = layers.GlobalAveragePooling2D()(x)
    
    # Custom classifier
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='FeatureExtraction')
    return model

# Create model
feature_model = create_feature_extraction_model(base_model)

# Count trainable parameters
trainable_params = sum([tf.size(w).numpy() for w in feature_model.trainable_weights])
total_params = feature_model.count_params()

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
# Train feature extraction model
print("Training with feature extraction...\n")

feature_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_feature = feature_model.fit(
    x_train_small, y_train_small,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate
test_loss, test_acc = feature_model.evaluate(x_test_resized, y_test, verbose=0)
print(f"\nFeature Extraction Test Accuracy: {test_acc:.4f}")

## 3. Fine-tuning Pre-trained Models

### Fine-tuning Strategy

**Steps:**
1. Start with feature extraction (train classifier)
2. Unfreeze top layers of base model
3. Continue training with very small learning rate
4. Monitor for overfitting

**Best Practices:**
- Use small learning rate (1e-5 to 1e-4)
- Unfreeze gradually (top layers first)
- More data = more layers to unfreeze
- Monitor validation metrics closely

### Which Layers to Unfreeze?

**General rule:**
- Early layers: Generic features (edges, textures)
- Later layers: Task-specific features
- Unfreeze later layers first

In [ ]:
# Fine-tuning: Unfreeze top layers
print("Setting up fine-tuning...\n")

# Unfreeze the base model
base_model.trainable = True

# Freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Count trainable layers
trainable_layers = sum([layer.trainable for layer in base_model.layers])
print(f"Total layers in base model: {len(base_model.layers)}")
print(f"Trainable layers: {trainable_layers}")
print(f"Frozen layers: {len(base_model.layers) - trainable_layers}")

In [ ]:
# Recompile with lower learning rate
feature_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # Much smaller learning rate
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nFine-tuning the model...\n")

# Continue training (fine-tuning)
history_finetune = feature_model.fit(
    x_train_small, y_train_small,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate after fine-tuning
test_loss, test_acc = feature_model.evaluate(x_test_resized, y_test, verbose=0)
print(f"\nFine-tuned Test Accuracy: {test_acc:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Combine histories
feature_epochs = len(history_feature.history['loss'])
finetune_epochs = len(history_finetune.history['loss'])
total_epochs = feature_epochs + finetune_epochs

# Loss
axes[0].plot(range(1, feature_epochs + 1), 
            history_feature.history['loss'], 
            label='Feature Extraction (train)', linewidth=2)
axes[0].plot(range(1, feature_epochs + 1), 
            history_feature.history['val_loss'], 
            label='Feature Extraction (val)', linewidth=2, linestyle='--')
axes[0].plot(range(feature_epochs + 1, total_epochs + 1), 
            history_finetune.history['loss'], 
            label='Fine-tuning (train)', linewidth=2)
axes[0].plot(range(feature_epochs + 1, total_epochs + 1), 
            history_finetune.history['val_loss'], 
            label='Fine-tuning (val)', linewidth=2, linestyle='--')
axes[0].axvline(x=feature_epochs, color='red', linestyle=':', linewidth=2, label='Start Fine-tuning')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss: Feature Extraction → Fine-tuning', 
                 fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(range(1, feature_epochs + 1), 
            history_feature.history['accuracy'], 
            label='Feature Extraction (train)', linewidth=2)
axes[1].plot(range(1, feature_epochs + 1), 
            history_feature.history['val_accuracy'], 
            label='Feature Extraction (val)', linewidth=2, linestyle='--')
axes[1].plot(range(feature_epochs + 1, total_epochs + 1), 
            history_finetune.history['accuracy'], 
            label='Fine-tuning (train)', linewidth=2)
axes[1].plot(range(feature_epochs + 1, total_epochs + 1), 
            history_finetune.history['val_accuracy'], 
            label='Fine-tuning (val)', linewidth=2, linestyle='--')
axes[1].axvline(x=feature_epochs, color='red', linestyle=':', linewidth=2, label='Start Fine-tuning')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy: Feature Extraction → Fine-tuning', 
                 fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Comparing Different Pre-trained Models

### Model Selection Guide

| Model | Parameters | Speed | Accuracy | Best For |
|-------|-----------|-------|----------|----------|
| **MobileNetV2** | 3.5M | Fast | Good | Mobile, edge devices |
| **ResNet50** | 25M | Medium | Very Good | General purpose |
| **VGG16** | 138M | Slow | Good | Feature extraction |
| **EfficientNetB0** | 5.3M | Fast | Excellent | Best efficiency |
| **ResNet101** | 44M | Slow | Excellent | High accuracy needed |

In [ ]:
# Compare different pre-trained models
def create_transfer_model(base_model_class, input_shape=(96, 96, 3), num_classes=10):
    """Create transfer learning model with any base"""
    base = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model

# Create models with different bases
models_comparison = {
    'MobileNetV2': MobileNetV2,
    'ResNet50': ResNet50,
    'EfficientNetB0': EfficientNetB0
}

print("Model Comparison:")
print("=" * 70)
for name, base_class in models_comparison.items():
    model = create_transfer_model(base_class)
    params = model.count_params()
    print(f"{name:20} | Parameters: {params:,}")
print("=" * 70)

## 5. Data Augmentation for Transfer Learning

### Why Augmentation Matters More

With transfer learning:
- Often have limited data
- Pre-trained on different domain
- Augmentation helps adaptation

**Recommended augmentations:**
- Random flips
- Random rotations (small angles)
- Random zoom
- Color jittering
- Random crops

In [ ]:
# Create data augmentation pipeline
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1)
])

# Model with augmentation
def create_augmented_transfer_model(base_model, num_classes=10):
    inputs = keras.Input(shape=(96, 96, 3))
    
    # Data augmentation (only during training)
    x = data_augmentation(inputs)
    
    # Pre-trained base
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model

print("Model with data augmentation created")

## 6. Practical Tips for Transfer Learning

### Decision Flow

**Small dataset, similar to ImageNet:**
- Use feature extraction only
- Don't fine-tune

**Small dataset, different from ImageNet:**
- Use feature extraction
- Fine-tune top layers carefully

**Large dataset, similar to ImageNet:**
- Start with feature extraction
- Fine-tune many layers

**Large dataset, different from ImageNet:**
- Fine-tune entire network
- Or train from scratch

### Common Mistakes to Avoid

1. **High learning rate during fine-tuning** → Destroys pre-trained weights
2. **Unfreezing too early** → Unstable training
3. **Wrong input preprocessing** → Poor performance
4. **Not using data augmentation** → Overfitting
5. **Forgetting to set training=False** → Batch norm issues

---

## 7. Multitask Learning (COMPSCI 714 Lecture 7-2)

### Core Idea

**Multitask Learning** trains a single model on multiple related tasks simultaneously. The multi-layer structure of neural networks makes them naturally suited for this.

**Architecture options:**
- **Shared bottom layers** + task-specific top layers (most common)
- **Fully separate** networks per task (no sharing)

### Real-World Example: Multilingual Speech Recognition

From COMPSCI 714 Lecture 7-2 (Huang et al., ICASSP 2013):
- Train a single model on French, German, Spanish, Italian, and Mandarin simultaneously
- Shared bottom layers capture **common acoustic features** across languages
- Task-specific top layers predict language-specific phoneme states
- **Result:** Training Mandarin with European language data significantly reduces character error rate, especially when Mandarin data is scarce

> 💡 **Key insight:** Human languages share common characteristics — multitask learning exploits this.

In [ ]:
# Multitask Learning: Shared encoder with task-specific heads
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

def create_multitask_model(input_dim=128, shared_dim=64, num_tasks=3, num_classes_per_task=10):
    """
    Multitask model: shared encoder + task-specific heads.
    
    Architecture (COMPSCI 714 Lecture 7-2):
      Input → Shared Layers → [Task A Head, Task B Head, Task C Head]
    """
    inputs = keras.Input(shape=(input_dim,), name='shared_input')
    
    # Shared bottom layers (learn common representations)
    x = layers.Dense(256, activation='relu', name='shared_1')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(shared_dim, activation='relu', name='shared_2')(x)
    shared_repr = layers.BatchNormalization(name='shared_repr')(x)
    
    # Task-specific heads
    task_outputs = []
    for i in range(num_tasks):
        task_x = layers.Dense(32, activation='relu', name=f'task_{i+1}_hidden')(shared_repr)
        task_out = layers.Dense(num_classes_per_task, activation='softmax',
                                name=f'task_{i+1}_output')(task_x)
        task_outputs.append(task_out)
    
    model = Model(inputs=inputs, outputs=task_outputs, name='MultitaskModel')
    return model

# Build and summarize
mt_model = create_multitask_model(num_tasks=3)
mt_model.compile(
    optimizer='adam',
    loss=['sparse_categorical_crossentropy'] * 3,
    loss_weights=[1.0, 1.0, 1.0],  # Equal weight for all tasks
    metrics=['accuracy']
)

print("Multitask Model Architecture:")
print(f"  Shared parameters: {sum(v.numpy().size for v in mt_model.trainable_variables[:6]):,}")
print(f"  Total parameters: {mt_model.count_params():,}")
print(f"  Number of output heads: 3")
print("\nKey benefit: shared layers learn representations useful for ALL tasks")
print("This is why multilingual models improve low-resource language performance!")

---

## 8. Domain-Adversarial Training (COMPSCI 714 Lecture 7-2)

### Problem: Domain Shift

When source and target domains differ (e.g., MNIST digits vs MNIST-M coloured digits), a model trained on source data may fail on target data because the **feature distributions are different**.

**Scenario:**
- Source data (xˢ, yˢ): labelled training data (e.g., black-and-white MNIST)
- Target data (xᵗ): **unlabelled** testing data from a different domain (e.g., coloured MNIST-M)
- Same task, different domains

### Solution: Domain-Adversarial Training

The key idea is to learn **domain-invariant features** — features that look the same regardless of whether the input came from the source or target domain.

**Architecture (3 components):**
1. **Feature Extractor** (shared): Extracts features from input
2. **Label Predictor**: Predicts the class label (maximise accuracy)
3. **Domain Classifier**: Predicts which domain the input came from (minimise accuracy)

**Training objective:**
```
Maximize label classification accuracy + Minimize domain classification accuracy
```

The feature extractor is trained to **fool the domain classifier** (adversarial) while still being useful for label prediction. This forces it to learn domain-invariant features.

> 📖 **Reference:** Ganin & Lempitsky (2015), "Unsupervised Domain Adaptation by Backpropagation", ICML

In [ ]:
# Domain-Adversarial Training: Architecture demonstration
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import numpy as np

@tf.custom_gradient
def gradient_reversal(x):
    """Gradient Reversal Layer (GRL).
    
    Forward pass: identity (x → x)
    Backward pass: negates gradient (−λ * grad)
    
    This is the key trick in domain-adversarial training:
    the feature extractor is trained to FOOL the domain classifier.
    """
    def grad(dy):
        return -dy  # Reverse gradient
    return x, grad

class GradientReversalLayer(layers.Layer):
    """Gradient Reversal Layer for domain-adversarial training."""
    def call(self, x):
        return gradient_reversal(x)

def create_domain_adversarial_model(input_dim=256, num_classes=10):
    """
    Domain-Adversarial Neural Network (DANN).
    
    Architecture (COMPSCI 714 Lecture 7-2):
      Input → Feature Extractor → [Label Predictor, Domain Classifier (via GRL)]
    
    Goal: Maximize label accuracy + Minimize domain classification accuracy
    """
    inputs = keras.Input(shape=(input_dim,), name='input')
    
    # Feature Extractor (shared)
    feat = layers.Dense(128, activation='relu', name='feat_1')(inputs)
    feat = layers.Dense(64, activation='relu', name='feat_2')(feat)
    features = layers.Dense(32, activation='relu', name='features')(feat)
    
    # Label Predictor (standard forward pass)
    label_out = layers.Dense(64, activation='relu', name='label_hidden')(features)
    label_pred = layers.Dense(num_classes, activation='softmax', name='label_pred')(label_out)
    
    # Domain Classifier (with Gradient Reversal Layer)
    # GRL makes feature extractor learn domain-INVARIANT features
    reversed_features = GradientReversalLayer(name='gradient_reversal')(features)
    domain_hidden = layers.Dense(32, activation='relu', name='domain_hidden')(reversed_features)
    domain_pred = layers.Dense(2, activation='softmax', name='domain_pred')(domain_hidden)
    
    model = Model(
        inputs=inputs,
        outputs=[label_pred, domain_pred],
        name='DomainAdversarialNN'
    )
    return model

dann = create_domain_adversarial_model()
dann.compile(
    optimizer='adam',
    loss={
        'label_pred': 'sparse_categorical_crossentropy',
        'domain_pred': 'sparse_categorical_crossentropy'
    },
    loss_weights={
        'label_pred': 1.0,
        'domain_pred': -1.0  # Negative weight = minimise domain accuracy
    },
    metrics=['accuracy']
)

print("Domain-Adversarial Neural Network (DANN):")
print(f"  Total parameters: {dann.count_params():,}")
print("\nComponents:")
print("  1. Feature Extractor: learns domain-invariant representations")
print("  2. Label Predictor:   maximises task accuracy (standard)")
print("  3. Domain Classifier: tries to distinguish source vs target")
print("  4. Gradient Reversal: makes feature extractor FOOL domain classifier")
print("\nResult: features that are useful for the task but indistinguishable across domains")

---

## 9. Zero-shot and Few-shot Learning (COMPSCI 714 Lecture 7-2)

### Zero-shot Learning

**Task:** Classify images **without having seen any samples** from those categories during training.

**Goal:** Generalise to new, unseen categories based on learned relationships and features.

**Key insight:** Instead of predicting the label directly, **learn a similarity function**!

**Example (from lecture):** Someone who has seen a horse but doesn't know zebras could likely recognise one knowing that a zebra looks like a *horse with black and white stripes*. Zero-shot learning assumes a **semantic relationship** between seen and unseen classes.

**How it works:**
1. Train a similarity function `sim(x, x')` from a large-scale training dataset
2. At test time, compare the query with every sample in the support set
3. Find the sample with the highest similarity score → that's the predicted class

### Few-shot Learning

The model has access to a **very limited number of samples** during training (typically a few).

**Terminology:**
- **k-way:** The support set has k classes
- **n-shot:** Every class has n samples
- **Support set:** The small labelled set used for reference
- **Query:** The unlabelled sample to classify

**Example:** A 4-way 2-shot problem has 4 classes × 2 samples = 8 support set images.

**Solutions:**
- Meta-learning (learn to learn)
- Data augmentation
- Similarity-based methods (Prototypical Networks, Siamese Networks)

In [ ]:
# Zero-shot / Few-shot Learning: Similarity-based approach
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

def create_siamese_similarity_model(input_dim=128, embed_dim=64):
    """
    Siamese Network for learning a similarity function.
    
    COMPSCI 714 Lecture 7-2:
    'Instead of predicting the label, learning similarity!'
    
    - First, a similarity function is trained from large-scale training dataset.
    - Then, apply the similarity function for prediction:
        * Compare the query with every sample in the support set
        * Find the sample with the highest similarity score
    """
    # Shared encoder (same weights for both inputs)
    encoder_input = keras.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(encoder_input)
    x = layers.Dense(128, activation='relu')(x)
    embedding = layers.Dense(embed_dim, activation=None)(x)
    # L2 normalise embeddings for cosine similarity
    embedding = layers.Lambda(lambda e: tf.math.l2_normalize(e, axis=1))(embedding)
    encoder = Model(encoder_input, embedding, name='encoder')
    
    # Siamese inputs
    input_a = keras.Input(shape=(input_dim,), name='query')
    input_b = keras.Input(shape=(input_dim,), name='support')
    
    # Shared encoder
    embed_a = encoder(input_a)
    embed_b = encoder(input_b)
    
    # Cosine similarity score
    similarity = layers.Dot(axes=1, normalize=False, name='similarity')([embed_a, embed_b])
    
    siamese = Model(inputs=[input_a, input_b], outputs=similarity, name='SiameseNet')
    return siamese, encoder

def few_shot_predict(encoder, query, support_set, support_labels):
    """
    Few-shot prediction using learned similarity.
    
    Args:
        encoder: Trained embedding network
        query: Single query sample to classify
        support_set: k-way n-shot support samples
        support_labels: Labels for support set
    
    Returns:
        Predicted label (highest similarity)
    """
    query_emb = encoder(query[np.newaxis])
    support_embs = encoder(support_set)
    
    # Cosine similarities
    similarities = tf.reduce_sum(query_emb * support_embs, axis=1).numpy()
    
    # Predict: class of most similar support sample
    best_idx = np.argmax(similarities)
    return support_labels[best_idx], similarities

# Demo
siamese_model, encoder = create_siamese_similarity_model()
siamese_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Simulate a 4-way 1-shot scenario
np.random.seed(42)
support_set = np.random.randn(4, 128).astype('float32')  # 4 classes, 1 shot each
support_labels = ['cat', 'dog', 'horse', 'zebra']
query_sample = np.random.randn(128).astype('float32')

predicted_class, sims = few_shot_predict(encoder, query_sample, support_set, support_labels)

print("4-way 1-shot Few-shot Learning Demo:")
print(f"  Support set classes: {support_labels}")
print(f"  Similarity scores:")
for label, sim in zip(support_labels, sims):
    print(f"    {label:10}: {sim:.4f}")
print(f"  Predicted class: {predicted_class} (highest similarity)")
print("\nNote: encoder weights are random here — in practice, train on large dataset first!")

---

## 10. Self-supervised Learning (COMPSCI 714 Lecture 7-2)

### The Bridge to Foundation Models

When we don't have labelled data in the source domain, we can still use **unlabelled data** through self-supervised learning.

**Key idea:** Define a training task where the **data itself provides the supervision signal**.

**NLP Examples:**
- **Language Modeling (GPT):** Predict the next word given previous words
- **Masked Language Modeling (BERT):** Predict randomly masked words from context

**Workflow:**
1. Download millions of documents from the web (no labels needed)
2. Define a self-supervised task (e.g., predict masked tokens)
3. Train a large model on this task → **Pretrained Foundation Model**
4. Fine-tune on small labelled target data for downstream tasks

**This is exactly how BERT, GPT, and T5 work** — covered in detail in [I09 - Advanced Transformer Architectures](./I09%20-%20Advanced%20Transformer%20Architectures.ipynb).

| Method | Source Data | Target Data | Approach |
|--------|------------|-------------|----------|
| Model Fine-tuning | Labelled | Labelled | Train on source, fine-tune on target |
| Multitask Learning | Labelled | Labelled | Train on all tasks jointly |
| Domain-Adversarial | Labelled | Unlabelled | Align feature distributions |
| Zero/Few-shot | Labelled | Unlabelled | Learn similarity function |
| Self-supervised | **Unlabelled** | Labelled | Self-supervised pretraining + fine-tune |
| Self-taught Clustering | **Unlabelled** | Unlabelled | Unsupervised feature learning |

## Summary

In this lesson, you learned:

1. ✅ **Formal definition** of transfer learning (source/target domains and tasks — COMPSCI 714)
2. ✅ **Transfer learning taxonomy**: Model Fine-tuning, Conservation Learning, Layer Transfer, Multitask Learning, Domain-Adversarial Training, Zero/Few-shot Learning, Self-supervised Learning
3. ✅ **Conservation Learning**: constraining output and parameter differences to prevent overfitting
4. ✅ **Layer Transfer**: which layers to copy for speech vs images
5. ✅ **Multitask Learning**: shared representations across related tasks
6. ✅ **Domain-Adversarial Training**: aligning source and target feature distributions
7. ✅ **Zero-shot Learning**: learning similarity functions to classify unseen categories
8. ✅ **Few-shot Learning**: k-way n-shot support sets with meta-learning
9. ✅ Feature extraction and fine-tuning with pre-trained CNN models

**Key Takeaways:**
- Transfer learning is formally defined over source/target domains and tasks
- The right method depends on whether your data is labelled and how much you have
- Conservation Learning prevents overfitting when target data is scarce
- Domain-Adversarial Training aligns feature distributions across domains
- Zero-shot learning learns similarity, not labels — enabling unseen class recognition
- Self-supervised learning (BERT, GPT) is the foundation of modern pretrained models

**Next Steps:**
- Move to [I05b - Pretrained Foundation Models in CV](./I05b%20-%20Pretrained%20Foundation%20Models%20in%20CV.ipynb)
- Move to I06 - Object Detection and Segmentation
- See I09 for BERT, GPT, T5 pretrained models in NLP

---

**References (COMPSCI 714 Lecture 7-2):**
- Yosinski et al. (2014): "How transferable are features in deep neural networks?" NIPS
- Ganin & Lempitsky (2015): "Unsupervised Domain Adaptation by Backpropagation" ICML
- Ajakan et al. (2016): "Domain-Adversarial Training of Neural Networks" JMLR
- Raina et al. (2007): "Self-taught learning: transfer learning from unlabeled data" ICML
- Huang et al. (2013): "Cross-language knowledge transfer using multilingual deep neural network with shared hidden layers" ICASSP

---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I05)
- [ ] Pre-trained models
- [ ] Fine-tuning strategies
- [ ] Domain adaptation
- [ ] Few-shot learning

---